# 📝 SQL + SQLite + PostgreSQL
### Exercises & Solutions — 28 Problems

This notebook is exercises-and-solutions only. It assumes you've already covered the
concept notebook for this topic. Each problem targets a **distinct function, pattern,
or real-world scenario** so that working through all of them gives you practical
exposure to everything commonly used on the job.

**Coverage map:**

- Schema design & basic CRUD (1-5)
- Filtering, sorting, aggregation (6-10)
- JOINs: inner, left, self, multi-table (11-15)
- Subqueries, CTEs, window functions (16-20)
- Transactions, constraints, indexes (21-24)
- Python integration patterns: repository, safe dynamic queries, bulk ops (25-28)

**Note:** All exercises run live against real SQLite databases (stdlib `sqlite3`,
no setup needed). PostgreSQL-specific syntax differences are called out in comments
where relevant, since this environment has no live Postgres server.


---


### 1. Create Table with Constraints

Create a `products` table with `PRIMARY KEY`, `NOT NULL`, `UNIQUE`, `CHECK`, and `DEFAULT` constraints all in one schema.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.executescript("""
CREATE TABLE products (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sku TEXT NOT NULL UNIQUE,
    name TEXT NOT NULL,
    price REAL NOT NULL CHECK (price > 0),
    in_stock INTEGER NOT NULL DEFAULT 1
);
""")
print("Table created successfully")
cur.execute("PRAGMA table_info(products)")
for row in cur.fetchall():
    print(f"  {row['name']}: {row['type']} (notnull={row['notnull']}, default={row['dflt_value']})")

### 2. INSERT with Parameterized Queries (Multiple Rows)

Insert 5 products using `executemany` with parameterized placeholders, then verify the CHECK constraint rejects a negative price.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("""
CREATE TABLE products (id INTEGER PRIMARY KEY, sku TEXT UNIQUE, name TEXT, price REAL CHECK (price > 0))
""")
products = [("SKU1","Widget",9.99), ("SKU2","Gadget",19.99), ("SKU3","Gizmo",29.99),
            ("SKU4","Doohickey",4.99), ("SKU5","Thingamajig",14.99)]
cur.executemany("INSERT INTO products (sku, name, price) VALUES (?, ?, ?)", products)
conn.commit()
print(f"Inserted {cur.rowcount if cur.rowcount > 0 else len(products)} rows")

try:
    cur.execute("INSERT INTO products (sku, name, price) VALUES (?, ?, ?)", ("BAD", "Broken", -5))
except sqlite3.IntegrityError as e:
    print(f"CHECK constraint blocked it: {e}")

### 3. UPDATE with WHERE and Verifying Row Count

Update prices for products matching a condition, checking `cursor.rowcount` to confirm how many rows were affected.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL, category TEXT)")
cur.executemany("INSERT INTO products (name, price, category) VALUES (?,?,?)", [
    ("A", 10, "electronics"), ("B", 20, "electronics"), ("C", 15, "books"), ("D", 30, "electronics")
])
conn.commit()

cur.execute("UPDATE products SET price = price * 1.1 WHERE category = ?", ("electronics",))
conn.commit()
print(f"Updated {cur.rowcount} rows")

cur.execute("SELECT name, price FROM products ORDER BY name")
for row in cur.fetchall():
    print(f"  {row['name']}: ${row['price']:.2f}")

### 4. DELETE with Subquery Condition

Delete all orders from customers who haven't placed an order in over a year (simulated), using a subquery in the DELETE's WHERE clause.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.executescript("""
CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, last_order_date TEXT);
CREATE TABLE carts (id INTEGER PRIMARY KEY, customer_id INTEGER, status TEXT);
""")
cur.executemany("INSERT INTO customers VALUES (?,?,?)", [
    (1, "Alice", "2024-01-01"), (2, "Bob", "2020-01-01"), (3, "Carol", "2023-06-01")
])
cur.executemany("INSERT INTO carts VALUES (?,?,?)", [(1,2,"abandoned"), (2,1,"abandoned"), (3,3,"abandoned")])
conn.commit()

cur.execute("""
    DELETE FROM carts WHERE customer_id IN (
        SELECT id FROM customers WHERE last_order_date < '2022-01-01'
    )
""")
conn.commit()
print(f"Deleted {cur.rowcount} stale cart(s)")
cur.execute("SELECT * FROM carts")
print("Remaining carts:", [dict(r) for r in cur.fetchall()])

### 5. UPSERT Pattern (INSERT ... ON CONFLICT)

Use SQLite's `ON CONFLICT ... DO UPDATE` (works the same in PostgreSQL) to implement an upsert — insert or update inventory counts.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE inventory (sku TEXT PRIMARY KEY, quantity INTEGER)")
cur.execute("INSERT INTO inventory VALUES ('ABC', 10)")
conn.commit()

def upsert_inventory(sku, add_qty):
    cur.execute("""
        INSERT INTO inventory (sku, quantity) VALUES (?, ?)
        ON CONFLICT(sku) DO UPDATE SET quantity = quantity + excluded.quantity
    """, (sku, add_qty))
    conn.commit()

upsert_inventory("ABC", 5)     # existing -> adds to current
upsert_inventory("XYZ", 20)    # new -> inserts fresh

cur.execute("SELECT * FROM inventory")
print([dict(r) for r in cur.fetchall()])

### 6. Filtering with AND/OR/NOT and IN

Write queries combining `AND`, `OR`, `NOT`, and `IN` to filter products by multiple business conditions in a single query.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL, category TEXT, active INTEGER)")
cur.executemany("INSERT INTO products (name,price,category,active) VALUES (?,?,?,?)", [
    ("A", 50, "electronics", 1), ("B", 150, "electronics", 1), ("C", 20, "books", 0),
    ("D", 80, "clothing", 1), ("E", 200, "electronics", 0),
])
conn.commit()

cur.execute("""
    SELECT name, price, category FROM products
    WHERE category IN ('electronics', 'clothing') AND active = 1 AND NOT price > 150
""")
print([dict(r) for r in cur.fetchall()])

### 7. LIKE Pattern Matching and BETWEEN Range Filtering

Use `LIKE` with wildcards for partial text matching, and `BETWEEN` for inclusive numeric ranges.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT, salary REAL)")
cur.executemany("INSERT INTO employees (name, salary) VALUES (?,?)", [
    ("Alice Anderson", 65000), ("Bob Smith", 85000), ("Anna Bell", 72000), ("Carl Davis", 95000)
])
conn.commit()

print("Names starting with 'A':")
cur.execute("SELECT name FROM employees WHERE name LIKE 'A%'")
print([r["name"] for r in cur.fetchall()])

print("\nSalaries between 70000 and 90000:")
cur.execute("SELECT name, salary FROM employees WHERE salary BETWEEN 70000 AND 90000")
print([dict(r) for r in cur.fetchall()])

### 8. ORDER BY Multiple Columns with Mixed Direction

Sort by department ASCENDING, then salary DESCENDING within each department — a common multi-column sort pattern.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE staff (name TEXT, dept TEXT, salary REAL)")
cur.executemany("INSERT INTO staff VALUES (?,?,?)", [
    ("Alice","Eng",90000), ("Bob","Eng",110000), ("Carol","Sales",75000),
    ("Dave","Sales",95000), ("Eve","Eng",100000)
])
conn.commit()

cur.execute("SELECT * FROM staff ORDER BY dept ASC, salary DESC")
for row in cur.fetchall():
    print(f"  {row['dept']}: {row['name']} (${row['salary']:,.0f})")

### 9. GROUP BY with Multiple Aggregate Functions

Compute COUNT, SUM, AVG, MIN, MAX all in one GROUP BY query for sales-by-region analysis.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE sales (region TEXT, amount REAL)")
cur.executemany("INSERT INTO sales VALUES (?,?)", [
    ("North", 500), ("South", 300), ("North", 700), ("East", 200),
    ("South", 600), ("North", 250), ("East", 450)
])
conn.commit()

cur.execute("""
    SELECT region, COUNT(*) as cnt, SUM(amount) as total, AVG(amount) as avg,
           MIN(amount) as min_sale, MAX(amount) as max_sale
    FROM sales GROUP BY region ORDER BY total DESC
""")
for row in cur.fetchall():
    print(f"  {row['region']}: count={row['cnt']}, total=${row['total']:.0f}, "
          f"avg=${row['avg']:.0f}, range=${row['min_sale']:.0f}-${row['max_sale']:.0f}")

### 10. HAVING to Filter Aggregated Groups

Find regions with average sale amount above 400 — demonstrating WHY this needs `HAVING` not `WHERE` (since it filters an aggregate).

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE sales (region TEXT, amount REAL)")
cur.executemany("INSERT INTO sales VALUES (?,?)", [
    ("North", 500), ("South", 300), ("North", 700), ("East", 200),
    ("South", 600), ("North", 250), ("East", 150)
])
conn.commit()

cur.execute("""
    SELECT region, AVG(amount) as avg_sale
    FROM sales GROUP BY region
    HAVING AVG(amount) > 400
""")
print("Regions with avg sale > 400:", [dict(r) for r in cur.fetchall()])

# Proving WHERE can't do this:
try:
    cur.execute("SELECT region FROM sales WHERE AVG(amount) > 400")
except sqlite3.OperationalError as e:
    print(f"\nWHERE with aggregate fails: {e}")

### 11. INNER JOIN Across Two Tables

Join `orders` to `customers` to show order details WITH the customer's name attached.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.executescript("""
CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL);
""")
cur.executemany("INSERT INTO customers VALUES (?,?)", [(1,"Alice"),(2,"Bob")])
cur.executemany("INSERT INTO orders VALUES (?,?,?)", [(1,1,100),(2,2,200),(3,1,150)])
conn.commit()

cur.execute("""
    SELECT o.id as order_id, c.name as customer, o.amount
    FROM orders o INNER JOIN customers c ON o.customer_id = c.id
""")
for row in cur.fetchall():
    print(dict(row))

### 12. LEFT JOIN to Find Unmatched Rows

Use LEFT JOIN + `WHERE right_table.id IS NULL` to find customers who have NEVER placed an order.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.executescript("""
CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER);
""")
cur.executemany("INSERT INTO customers VALUES (?,?)", [(1,"Alice"),(2,"Bob"),(3,"Carol")])
cur.executemany("INSERT INTO orders VALUES (?,?)", [(1,1),(2,1)])   # only Alice has orders
conn.commit()

cur.execute("""
    SELECT c.name FROM customers c
    LEFT JOIN orders o ON c.id = o.customer_id
    WHERE o.id IS NULL
""")
print("Customers with NO orders:", [r["name"] for r in cur.fetchall()])

### 13. Self-JOIN for Hierarchical Data

Build an `employees` table with a `manager_id` self-referencing column, then JOIN the table to itself to list each employee with their manager's name.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT, manager_id INTEGER)")
cur.executemany("INSERT INTO employees VALUES (?,?,?)", [
    (1, "CEO Carol", None), (2, "VP Alice", 1), (3, "VP Bob", 1),
    (4, "Eng Dave", 2), (5, "Eng Eve", 2),
])
conn.commit()

cur.execute("""
    SELECT e.name as employee, m.name as manager
    FROM employees e LEFT JOIN employees m ON e.manager_id = m.id
""")
for row in cur.fetchall():
    print(f"  {row['employee']} reports to {row['manager'] or '(nobody - top level)'}")

### 14. Three-Table JOIN

Join `orders`, `customers`, AND `products` (via an order_items junction table) into a single readable report.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.executescript("""
CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL);
CREATE TABLE order_items (customer_id INTEGER, product_id INTEGER, qty INTEGER);
""")
cur.executemany("INSERT INTO customers VALUES (?,?)", [(1,"Alice"),(2,"Bob")])
cur.executemany("INSERT INTO products VALUES (?,?,?)", [(1,"Widget",10),(2,"Gadget",25)])
cur.executemany("INSERT INTO order_items VALUES (?,?,?)", [(1,1,3),(1,2,1),(2,2,2)])
conn.commit()

cur.execute("""
    SELECT c.name as customer, p.name as product, oi.qty, p.price * oi.qty as line_total
    FROM order_items oi
    JOIN customers c ON oi.customer_id = c.id
    JOIN products p ON oi.product_id = p.id
    ORDER BY c.name
""")
for row in cur.fetchall():
    print(dict(row))

### 15. FULL OUTER JOIN Simulation (SQLite lacks native support)

SQLite has no native `FULL OUTER JOIN` — simulate it via `LEFT JOIN UNION RIGHT-as-LEFT JOIN`, a real workaround you'd need in practice.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.executescript("""
CREATE TABLE a (id INTEGER PRIMARY KEY, val TEXT);
CREATE TABLE b (id INTEGER PRIMARY KEY, val TEXT);
""")
cur.executemany("INSERT INTO a VALUES (?,?)", [(1,"a1"),(2,"a2"),(3,"a3")])
cur.executemany("INSERT INTO b VALUES (?,?)", [(2,"b2"),(3,"b3"),(4,"b4")])
conn.commit()

# FULL OUTER JOIN = LEFT JOIN UNION (RIGHT JOIN's unmatched rows)
cur.execute("""
    SELECT a.id as a_id, a.val as a_val, b.id as b_id, b.val as b_val
    FROM a LEFT JOIN b ON a.id = b.id
    UNION
    SELECT a.id as a_id, a.val as a_val, b.id as b_id, b.val as b_val
    FROM b LEFT JOIN a ON a.id = b.id
""")
for row in cur.fetchall():
    print(dict(row))
print("\n(Note: real PostgreSQL supports FULL OUTER JOIN natively, no UNION workaround needed)")

### 16. Correlated Subquery

Write a correlated subquery finding each employee's salary RANK relative to others in their SAME department (without window functions).

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE emp (name TEXT, dept TEXT, salary REAL)")
cur.executemany("INSERT INTO emp VALUES (?,?,?)", [
    ("Alice","Eng",100), ("Bob","Eng",90), ("Carol","Eng",110),
    ("Dave","Sales",80), ("Eve","Sales",95),
])
conn.commit()

cur.execute("""
    SELECT e1.name, e1.dept, e1.salary,
           (SELECT COUNT(*) FROM emp e2 WHERE e2.dept = e1.dept AND e2.salary > e1.salary) + 1 as rank_in_dept
    FROM emp e1
    ORDER BY e1.dept, rank_in_dept
""")
for row in cur.fetchall():
    print(dict(row))

### 17. CTE (WITH clause) for Readable Multi-Step Queries

Use a CTE to first compute department averages, then join back to find above-average earners — more readable than nested subqueries.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE emp (name TEXT, dept TEXT, salary REAL)")
cur.executemany("INSERT INTO emp VALUES (?,?,?)", [
    ("Alice","Eng",100), ("Bob","Eng",90), ("Carol","Eng",110),
    ("Dave","Sales",80), ("Eve","Sales",95),
])
conn.commit()

cur.execute("""
    WITH dept_avg AS (
        SELECT dept, AVG(salary) as avg_sal FROM emp GROUP BY dept
    )
    SELECT e.name, e.dept, e.salary, d.avg_sal
    FROM emp e JOIN dept_avg d ON e.dept = d.dept
    WHERE e.salary > d.avg_sal
""")
for row in cur.fetchall():
    print(dict(row))

### 18. Recursive CTE for Hierarchical Traversal

Use a RECURSIVE CTE to traverse the employee management hierarchy from exercise 13, listing each person's full reporting chain depth.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT, manager_id INTEGER)")
cur.executemany("INSERT INTO employees VALUES (?,?,?)", [
    (1, "CEO", None), (2, "VP-A", 1), (3, "VP-B", 1), (4, "Eng-1", 2), (5, "Eng-2", 4),
])
conn.commit()

cur.execute("""
    WITH RECURSIVE org_chart(id, name, depth) AS (
        SELECT id, name, 0 FROM employees WHERE manager_id IS NULL
        UNION ALL
        SELECT e.id, e.name, oc.depth + 1
        FROM employees e JOIN org_chart oc ON e.manager_id = oc.id
    )
    SELECT name, depth FROM org_chart ORDER BY depth, name
""")
for row in cur.fetchall():
    print(f"{'  ' * row['depth']}{row['name']} (depth {row['depth']})")

### 19. Window Function: RANK, ROW_NUMBER, DENSE_RANK Comparison

Compute all three ranking functions side by side to show the difference when there are TIES in the data.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE scores (name TEXT, score INTEGER)")
cur.executemany("INSERT INTO scores VALUES (?,?)", [
    ("Alice", 90), ("Bob", 85), ("Carol", 90), ("Dave", 80), ("Eve", 85),
])
conn.commit()

cur.execute("""
    SELECT name, score,
           ROW_NUMBER() OVER (ORDER BY score DESC) as row_num,
           RANK() OVER (ORDER BY score DESC) as rank,
           DENSE_RANK() OVER (ORDER BY score DESC) as dense_rank
    FROM scores ORDER BY score DESC
""")
for row in cur.fetchall():
    print(dict(row))
print("\nNote how RANK skips numbers after ties, DENSE_RANK doesn't, ROW_NUMBER never ties")

### 20. Window Function: Running Total with PARTITION BY

Compute a running total of sales PER region (partitioned), using `SUM() OVER (PARTITION BY ... ORDER BY ...)`.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE sales (region TEXT, day INTEGER, amount REAL)")
cur.executemany("INSERT INTO sales VALUES (?,?,?)", [
    ("North",1,100), ("North",2,150), ("North",3,80),
    ("South",1,200), ("South",2,50), ("South",3,120),
])
conn.commit()

cur.execute("""
    SELECT region, day, amount,
           SUM(amount) OVER (PARTITION BY region ORDER BY day) as running_total
    FROM sales ORDER BY region, day
""")
for row in cur.fetchall():
    print(dict(row))

### 21. Transaction Atomicity: All-or-Nothing Multi-Step Update

Implement a bank transfer (debit + credit) inside an explicit transaction, proving a failure mid-way rolls back BOTH operations.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance REAL)")
cur.executemany("INSERT INTO accounts VALUES (?,?)", [(1,100),(2,50)])
conn.commit()

def transfer(from_id, to_id, amount):
    cur.execute("SELECT balance FROM accounts WHERE id=?", (from_id,))
    if cur.fetchone()["balance"] < amount:
        raise ValueError("insufficient funds")
    cur.execute("UPDATE accounts SET balance = balance - ? WHERE id=?", (amount, from_id))
    cur.execute("UPDATE accounts SET balance = balance + ? WHERE id=?", (amount, to_id))

try:
    transfer(1, 2, 500)   # more than account 1 has
    conn.commit()
except ValueError as e:
    conn.rollback()
    print(f"Rolled back: {e}")

cur.execute("SELECT * FROM accounts")
print("Balances unchanged after rollback:", [dict(r) for r in cur.fetchall()])

### 22. FOREIGN KEY Constraint Enforcement

Enable `PRAGMA foreign_keys=ON` (off by default in SQLite!) and verify it correctly BLOCKS inserting an order for a nonexistent customer.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("PRAGMA foreign_keys = ON")   # SQLite requires this explicitly!
cur.executescript("""
CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER REFERENCES customers(id));
""")
cur.execute("INSERT INTO customers VALUES (1, 'Alice')")
conn.commit()

cur.execute("INSERT INTO orders VALUES (1, 1)")   # valid - customer exists
conn.commit()
print("Valid order inserted")

try:
    cur.execute("INSERT INTO orders VALUES (2, 999)")  # invalid - no such customer
    conn.commit()
except sqlite3.IntegrityError as e:
    print(f"FK constraint blocked it: {e}")

### 23. Measuring Index Impact on Query Performance

Create a large table WITHOUT an index, time a filtered query, then add an index on the filtered column and re-time — demonstrating real speedup.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

import time, random
random.seed(0)

cur.execute("CREATE TABLE big_table (id INTEGER PRIMARY KEY, lookup_key INTEGER, payload TEXT)")
rows = [(i, random.randint(0, 100000), f"payload-{i}") for i in range(50000)]
cur.executemany("INSERT INTO big_table (id, lookup_key, payload) VALUES (?,?,?)", rows)
conn.commit()

target = rows[25000][1]   # a real lookup_key value to search for

start = time.perf_counter()
cur.execute("SELECT * FROM big_table WHERE lookup_key = ?", (target,))
cur.fetchall()
no_index_time = time.perf_counter() - start

cur.execute("CREATE INDEX idx_lookup ON big_table(lookup_key)")
conn.commit()

start = time.perf_counter()
cur.execute("SELECT * FROM big_table WHERE lookup_key = ?", (target,))
cur.fetchall()
with_index_time = time.perf_counter() - start

print(f"Without index: {no_index_time*1000:.3f}ms")
print(f"With index:    {with_index_time*1000:.3f}ms")
print(f"Speedup: {no_index_time/with_index_time:.1f}x" if with_index_time > 0 else "negligible at this scale")

### 24. EXPLAIN QUERY PLAN to Verify Index Usage

Use `EXPLAIN QUERY PLAN` to CONFIRM whether SQLite's planner actually uses an index for a given query (don't just assume it does).

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, category TEXT, value INTEGER)")
cur.executemany("INSERT INTO t (category, value) VALUES (?,?)", [("A", i) for i in range(1000)])
conn.commit()

cur.execute("EXPLAIN QUERY PLAN SELECT * FROM t WHERE category = 'A'")
print("WITHOUT index:")
for row in cur.fetchall():
    print(" ", dict(row))

cur.execute("CREATE INDEX idx_cat ON t(category)")
conn.commit()

cur.execute("EXPLAIN QUERY PLAN SELECT * FROM t WHERE category = 'A'")
print("\nWITH index:")
for row in cur.fetchall():
    print(" ", dict(row))

### 25. Repository Pattern Wrapping Raw SQL

Build a small `UserRepository` class encapsulating ALL SQL for a `users` table, exposing clean Python methods (`create`, `get`, `find_by_email`) to callers.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

class UserRepository:
    def __init__(self, connection):
        self.conn = connection
        self.conn.execute("CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY, name TEXT, email TEXT UNIQUE)")
        self.conn.commit()

    def create(self, name, email):
        cur = self.conn.execute("INSERT INTO users (name, email) VALUES (?, ?)", (name, email))
        self.conn.commit()
        return cur.lastrowid

    def get(self, user_id):
        row = self.conn.execute("SELECT * FROM users WHERE id=?", (user_id,)).fetchone()
        return dict(row) if row else None

    def find_by_email(self, email):
        row = self.conn.execute("SELECT * FROM users WHERE email=?", (email,)).fetchone()
        return dict(row) if row else None

repo = UserRepository(conn)
uid = repo.create("Alice", "alice@example.com")
print(repo.get(uid))
print(repo.find_by_email("alice@example.com"))

### 26. Safe Dynamic Query Builder (Whitelisting Columns)

Build a `build_safe_filter_query(table, filters, allowed_columns)` that constructs a parameterized WHERE clause dynamically WITHOUT risking SQL injection, by whitelisting column names.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("CREATE TABLE items (id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL)")
cur.executemany("INSERT INTO items (name,category,price) VALUES (?,?,?)", [
    ("A","tools",10), ("B","tools",20), ("C","toys",15)
])
conn.commit()

ALLOWED_COLUMNS = {"category", "price", "name"}

def build_safe_filter_query(table, filters: dict, allowed_columns: set):
    bad_cols = set(filters) - allowed_columns
    if bad_cols:
        raise ValueError(f"Disallowed filter columns: {bad_cols}")
    if not filters:
        return f"SELECT * FROM {table}", ()
    clause = " AND ".join(f"{col} = ?" for col in filters)
    return f"SELECT * FROM {table} WHERE {clause}", tuple(filters.values())

query, params = build_safe_filter_query("items", {"category": "tools"}, ALLOWED_COLUMNS)
cur.execute(query, params)
print([dict(r) for r in cur.fetchall()])

try:
    build_safe_filter_query("items", {"id; DROP TABLE items": "x"}, ALLOWED_COLUMNS)
except ValueError as e:
    print(f"Blocked injection attempt: {e}")

### 27. Bulk Insert Performance: executemany vs Loop

Compare inserting 10,000 rows via a Python loop with individual `execute()` calls vs ONE `executemany()` call, measuring the real time difference.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

import time

cur.execute("CREATE TABLE bulk_test (id INTEGER PRIMARY KEY, val INTEGER)")
conn.commit()

N = 5000
data = [(i,) for i in range(N)]

start = time.perf_counter()
for val in data:
    cur.execute("INSERT INTO bulk_test (val) VALUES (?)", val)
conn.commit()
loop_time = time.perf_counter() - start

cur.execute("DELETE FROM bulk_test")
conn.commit()

start = time.perf_counter()
cur.executemany("INSERT INTO bulk_test (val) VALUES (?)", data)
conn.commit()
batch_time = time.perf_counter() - start

print(f"Individual execute() x{N}: {loop_time:.3f}s")
print(f"Single executemany():     {batch_time:.3f}s")
print(f"executemany is {loop_time/batch_time:.1f}x faster")

### 28. Context-Manager-Based Connection Handling for Automatic Cleanup

Build a reusable `db_session()` context manager handling commit-on-success / rollback-on-exception automatically, applied to a real multi-step operation.

In [ ]:
import sqlite3
from contextlib import contextmanager

@contextmanager
def db_session(db_path=":memory:"):
    connection = sqlite3.connect(db_path)
    connection.row_factory = sqlite3.Row
    try:
        yield connection
        connection.commit()
    except Exception:
        connection.rollback()
        raise
    finally:
        connection.close()

with db_session() as session:
    session.execute("CREATE TABLE logs (id INTEGER PRIMARY KEY, message TEXT)")
    session.execute("INSERT INTO logs (message) VALUES (?)", ("first entry",))
print("Session 1 committed successfully")

try:
    with db_session("/tmp/test_session.db") as session:
        session.execute("CREATE TABLE IF NOT EXISTS logs2 (id INTEGER PRIMARY KEY, message TEXT)")
        session.execute("INSERT INTO logs2 (message) VALUES (?)", ("will be rolled back",))
        raise RuntimeError("simulated failure mid-transaction")
except RuntimeError as e:
    print(f"Session 2 failed and rolled back: {e}")

# Verify rollback actually happened
check = sqlite3.connect("/tmp/test_session.db")
result = check.execute("SELECT COUNT(*) FROM logs2").fetchone()
print(f"Rows in logs2 after rollback: {result[0]} (should be 0)")
check.close()
import os; os.remove("/tmp/test_session.db")